In [1]:
import pandas as pd
from census import Census
from pygris import tracts
import geopandas as gpd
from tobler.area_weighted import area_interpolate
import numpy as np

Pull from census API

In [1]:
# Replace with your API key
CENSUS_API_KEY = "YOUR API KEY"

c = Census(CENSUS_API_KEY, year=2024)

vars = [
    'B25002_001E',  # Total housing units
    'B25002_002E',  # Total occupied housing units
    'B25002_003E',  # Total vacant housing units
    'B25003_002E',  # Owner occupied units
    'B25003_003E',  # Renter occupied units
    'B25064_001E',  # Median gross rent
    'B25077_001E',  # Median home value
    'B25004_003E',  # Vacant homes rented, not occupied
    'B25004_004E',  # Vacant homes sold, not occupied
    'B25004_005E',  # Vacant homes for seasonal, recreational, or occasional use
    'B19013_001E',  # Median household income
    'B01003_001E',  # Population
]

# Main pull
data = c.acs5.state_county_tract(('NAME', 'B25002_001E', 'B25002_002E', 'B25002_003E', 'B25003_002E', 'B25003_003E',
                                  'B25064_001E', 'B25077_001E', 'B25004_003E', 'B25004_004E', 'B25004_005E',
                                  'B19013_001E', 'B01003_001E'),
                                   '08', '031', Census.ALL)


# Convert to pandas DataFrame
df = pd.DataFrame(data)

NameError: name 'Census' is not defined

In [ ]:
df

,NAME,B25002_001E,B25002_002E,B25002_003E,B25003_002E,B25003_003E,B25064_001E,B25077_001E,B25004_003E,B25004_004E,B25004_005E,B19013_001E,B01003_001E,state,county,tract
0,Census Tract 1.02; Denver County; Colorado,1798.0,1691.0,107.0,1190.0,501.0,2540.0,904600.0,0.0,0.0,0.0,174271.0,3588.0,08,031,000102
1,Census Tract 2.01; Denver County; Colorado,1803.0,1795.0,8.0,1219.0,576.0,1291.0,535300.0,0.0,0.0,0.0,97639.0,3810.0,08,031,000201
2,Census Tract 2.02; Denver County; Colorado,2072.0,1881.0,191.0,1173.0,708.0,1539.0,708200.0,0.0,22.0,0.0,96740.0,3795.0,08,031,000202
3,Census Tract 3.01; Denver County; Colorado,2988.0,2853.0,135.0,1892.0,961.0,1686.0,918600.0,0.0,0.0,0.0,123665.0,5565.0,08,031,000301
4,Census Tract 3.02; Denver County; Colorado,2592.0,2549.0,43.0,1605.0,944.0,1857.0,712700.0,0.0,0.0,0.0,132940.0,4785.0,08,031,000302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,Census Tract 157; Denver County; Colorado,2422.0,2228.0,194.0,950.0,1278.0,1716.0,553800.0,39.0,0.0,0.0,84164.0,5778.0,08,031,015700
174,Census Tract 158; Denver County; Colorado,1911.0,1708.0,203.0,597.0,1111.0,1927.0,410000.0,0.0,0.0,0.0,87132.0,4011.0,08,031,015800
175,Census Tract 9800.01; Denver County; Colorado,0.0,0.0,0.0,0.0,0.0,-666666666.0,-666666666.0,0.0,0.0,0.0,-666666666.0,0.0,08,031,980001
176,Census Tract 9801; Denver County; Colorado,0.0,0.0,0.0,0.0,0.0,-666666666.0,-666666666.0,0.0,0.0,0.0,-666666666.0,0.0,08,031,980100


In [ ]:
# Load Denver County tract data
denver_tracts = tracts(state='CO', county='Denver', year=2024)

# Read Denver neighborhood boundaries
neighborhoods = gpd.read_file('/Users/evanfarrenkopf/Desktop/Projects/Airbnb/data/denver_neighborhoods.geojson')

# reproject to a projected CRS for accurate math
tracts_proj = denver_tracts.to_crs(epsg=3735)
neighborhoods_proj = neighborhoods.to_crs(epsg=3735)

Using FIPS code '08' for input 'CO'
Using FIPS code '031' for input 'Denver'


In [ ]:
df['GEOID'] = df['state'] + df['county'] + df['tract']

tracts_proj = tracts_proj.merge(df, on='GEOID')

# Remove tracts with bad/null data
bad_tracts = tracts_proj[(tracts_proj[vars] < 0).any(axis=1)]
tracts_proj = tracts_proj.drop(bad_tracts.index)
print(bad_tracts[['GEOID'] + vars])


           GEOID  B25002_001E  B25002_002E  B25002_003E  B25003_002E  \
7    08031004103       2438.0       2388.0         50.0       2174.0   
13   08031004308       2787.0       2527.0        260.0        833.0   
21   08031002704       2001.0       1877.0        124.0        183.0   
40   08031002604       1731.0       1596.0        135.0          0.0   
73   08031980001          0.0          0.0          0.0          0.0   
78   08031980100          0.0          0.0          0.0          0.0   
92   08031007006       2297.0       2128.0        169.0          0.0   
105  08031000800        477.0        429.0         48.0         35.0   
165  08031005503       1882.0       1813.0         69.0       1533.0   

     B25003_003E  B25064_001E  B25077_001E  B25004_003E  B25004_004E  \
7          214.0 -666666666.0     817800.0          0.0         50.0   
13        1694.0       1716.0 -666666666.0         96.0          0.0   
21        1694.0       1395.0 -666666666.0          0.0        

# Interpolate tract-level data to neighborhood boundaries

In [ ]:
extensive_vars = [
    'B25002_001E',  # Total housing units
    'B25002_002E',  # Total occupied housing units
    'B25002_003E',  # Total vacant housing units
    'B25003_002E',  # Owner occupied units
    'B25003_003E',  # Renter occupied units
    'B25004_003E',  # Vacant homes rented, not occupied
    'B25004_004E',  # Vacant homes sold, not occupied
    'B25004_005E',  # Vacant homes for seasonal, recreational, or occasional use
    'B01003_001E',  # Population
]

intensive_vars = [
    'B25064_001E',  # Median gross rent
    'B25077_001E',  # Median home value
    'B19013_001E',  # Median household income
]

interpolated = area_interpolate(
    source_df=tracts_proj,
    target_df=neighborhoods_proj,
    extensive_variables=extensive_vars,
    intensive_variables=intensive_vars
)

interpolated = interpolated.rename(columns={
    'B25002_001E': 'total_housing',
    'B25002_002E': 'total_occupied_housing',
    'B25002_003E': 'total_vacant_housing',
    'B25003_002E': 'owner_occupied_units',
    'B25003_003E': 'renter_occupied_units',
    'B25064_001E': 'median_gross_rent',
    'B25077_001E': 'median_home_value',
    'B25004_003E': 'vacant_homes_rented',
    'B25004_004E': 'vacant_homes_sold',
    'B25004_005E': 'vacant_homes_seasonal',
    'B19013_001E': 'median_household_income',
    'B01003_001E': 'population'
})

interpolated.insert(0, 'neighborhood', neighborhoods_proj['NBHD_NAME'].values)

interpolated = interpolated.round(0)

In [ ]:
interpolated

,neighborhood,total_housing,total_occupied_housing,total_vacant_housing,owner_occupied_units,renter_occupied_units,vacant_homes_rented,vacant_homes_sold,vacant_homes_seasonal,population,median_gross_rent,median_home_value,median_household_income,geometry
0,Marston,5614.0,5487.0,127.0,3330.0,2156.0,0.0,0.0,0.0,11635.0,1964.0,550295.0,87935.0,"POLYGON ((-4322130.826 1379611.278, -4321997.8..."
1,Fort Logan,1935.0,1741.0,194.0,1283.0,458.0,0.0,0.0,0.0,4232.0,1927.0,665828.0,106371.0,"POLYGON ((-4310689.931 1389758.903, -4311031.4..."
2,Bear Valley,3906.0,3636.0,269.0,2380.0,1256.0,0.0,0.0,0.0,8161.0,1724.0,555477.0,85977.0,"POLYGON ((-4321416.247 1398528.763, -4321366.4..."
3,Harvey Park South,3626.0,3396.0,230.0,1912.0,1484.0,39.0,0.0,0.0,8841.0,2093.0,553221.0,94378.0,"POLYGON ((-4306797.6 1394194.872, -4306867.676..."
4,Southmoor Park,3957.0,3627.0,330.0,896.0,2731.0,53.0,0.0,0.0,6042.0,2135.0,717430.0,111692.0,"POLYGON ((-4277073.627 1370250.077, -4277667.9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,Central Park,12579.0,11774.0,805.0,8290.0,3484.0,1.0,39.0,0.0,32844.0,2140.0,875838.0,192082.0,"POLYGON ((-4251723.149 1429331.245, -4251726.7..."
74,Montbello,9445.0,9108.0,337.0,5884.0,3224.0,86.0,28.0,0.0,32422.0,1914.0,422651.0,88644.0,"POLYGON ((-4238708.9 1416434.485, -4241260.361..."
75,Lowry Field,5908.0,5583.0,325.0,2819.0,2764.0,10.0,44.0,2.0,11150.0,2294.0,763745.0,112991.0,"POLYGON ((-4262667.178 1407465.673, -4262874.3..."
76,Virginia Village,7527.0,7274.0,254.0,3190.0,4083.0,53.0,61.0,0.0,14432.0,1988.0,634666.0,108585.0,"MULTIPOLYGON (((-4279762.498 1398644.575, -427..."


In [ ]:
airbnb_data = pd.read_csv('/Users/evanfarrenkopf/Desktop/Projects/Airbnb/data/neighborhoods.csv', sep='\t')
airbnb_data

,neighborhood,total_listings,pct_entire_home,avg_price,avg_bedrooms,avg_price_per_bedroom,avg_estimated_annual_revenue,estimated_total_annual_revenue,total_hosts,hobbyist_hosts,small_business_hosts,commercial_hosts
0,Athmar Park,66,87.88,205.59,2.49,68.98,16446.43,838768.0,45,32,10,3
1,Auraria,5,100.00,229.72,1.80,127.62,27151.80,135759.0,5,3,1,1
2,Baker,116,88.79,189.34,1.69,105.31,25062.80,2355903.0,89,64,16,9
3,Barnum,33,96.97,197.91,2.17,88.30,18324.21,531402.0,27,22,2,3
4,Barnum West,18,94.44,201.42,2.50,58.19,15947.08,207312.0,17,13,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...
72,West Colfax,195,93.33,266.70,2.17,108.33,33784.39,5337933.0,157,123,20,14
73,West Highland,147,89.80,234.03,2.05,97.58,26283.93,3101504.0,122,94,12,16
74,Westwood,27,100.00,186.40,2.00,86.30,22644.48,566112.0,21,17,3,1
75,Whittier,108,82.41,264.19,1.93,127.10,34169.30,3041068.0,90,67,14,9


In [ ]:
neighborhoods = interpolated.merge(airbnb_data, on='neighborhood')
neighborhoods.columns

Index(['neighborhood', 'total_housing', 'total_occupied_housing',
       'total_vacant_housing', 'owner_occupied_units', 'renter_occupied_units',
       'vacant_homes_rented', 'vacant_homes_sold', 'vacant_homes_seasonal',
       'population', 'median_gross_rent', 'median_home_value',
       'median_household_income', 'geometry', 'total_listings',
       'pct_entire_home', 'avg_price', 'avg_bedrooms', 'avg_price_per_bedroom',
       'avg_estimated_annual_revenue', 'estimated_total_annual_revenue',
       'total_hosts', 'hobbyist_hosts', 'small_business_hosts',
       'commercial_hosts'],
      dtype='str')

In [ ]:
print(neighborhoods[['neighborhood', 'geometry']])

                neighborhood  \
0                    Marston   
1                 Fort Logan   
2                Bear Valley   
3          Harvey Park South   
4             Southmoor Park   
..                       ...   
71               Five Points   
72                 Montbello   
73               Lowry Field   
74          Virginia Village   
75  Washington Virginia Vale   

                                             geometry  
0   POLYGON ((-4322130.826 1379611.278, -4321997.8...  
1   POLYGON ((-4310689.931 1389758.903, -4311031.4...  
2   POLYGON ((-4321416.247 1398528.763, -4321366.4...  
3   POLYGON ((-4306797.6 1394194.872, -4306867.676...  
4   POLYGON ((-4277073.627 1370250.077, -4277667.9...  
..                                                ...  
71  POLYGON ((-4283082.58 1426718.063, -4283572.38...  
72  POLYGON ((-4238708.9 1416434.485, -4241260.361...  
73  POLYGON ((-4262667.178 1407465.673, -4262874.3...  
74  MULTIPOLYGON (((-4279762.498 1398644.575, -427...  

In [ ]:
neighborhoods['geometry'] = neighborhoods['geometry'].astype(str)

/var/folders/ty/9_352qyj78zbb8vgbqvdnf_m0000gn/T/ipykernel_31890/2944553894.py:1: UserWarning: Geometry column does not contain geometry.
  neighborhoods['geometry'] = neighborhoods['geometry'].astype(str)


In [ ]:
neighborhoods.to_csv('data/neighborhoods_complete.csv', index=False)